# Guaranteed k-anonymity

This notebook shows the first proper contribution - `guaranteed_k_anonymity` parameter.
We show how increasing the minimum number of samples in the initial DT leaves, can lead to faster & better generalization and also to potentially better accuracy.

Interpretation:

The original codebase hardcodes the surrogate decision tree to split until it reaches perfectly homogeneous leaves, which often results in hyper-specific clusters containing only a single individual. Then, it's the job of the uprooting algorithm to increase the generalizations by pruning these highly specific leaves. This however, is not guaranteed to happen.

We introduce the `min_samples_leaf=k` parameter to shift the algorithm from a "reactive" privacy approach to a "security-by-design" approach. This directly addresses the authors' own suggestion to explore "the number of samples in each leaf" to optimize generalizations.

**Pros**
- **Guarantees k-Anonymity & Lowers Disclosure Risk**: It establishes a strict privacy floor. By guaranteeing that every generalized data point belongs to a cluster of at least k data points, it automatically masks unique individuals in a crowd, drastically reducing the dataset's identity disclosure risk.
- **Reduces Overfitting**: The original unrestricted tree massively overfits the complex target model to achieve perfect homogeneity. By restricting leaf size, the tree learns broader, more robust decision boundaries. This can often lead to better target accuracy on the production data.
- **Prevents the "Initial Accuracy Dip"**: The authors noted that indiscriminately pruning an overfitted tree causes a "large initial dip in accuracy" for almost no privacy gain because it destroys highly specific splits that affect very few samples. By enforcing larger baseline clusters, we prevent the tree from memorizing those edge cases in the first place, smoothing out the pruning phase.

**Cons**
- **Sacrifices Perfect Homogeneity**: The surrogate decision tree is no longer allowed to split until every single record in a leaf perfectly aligns with the target model's original predictions.
- **Misclassification of Minority Records**: Because the enforced k-sized clusters will now inevitably contain mixed target predictions, the algorithm must select a representative median point based only on the majority class of that leaf. Consequently, the minority records within that cluster will receive an incorrect prediction, creating a direct and unavoidable trade-off between establishing a strict privacy floor and maintaining "perfect" baseline accuracy.

---

Just like in the `minimization_adult_new.ipynb` we will use the same base model (random forest) and the same dataset (Adults).

## Step 1: Load Data

In [41]:
from ucimlrepo import fetch_ucirepo

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from apt.minimization import GeneralizeToRepresentative

from apt.utils.metrics import calculate_disclosure_risk

In [42]:
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets["income"]

# Convert labels to integers
y = y.apply(lambda x: 1 if x.startswith("<=50K") else 0).astype(int)

# Only keep 10% of dataset for faster execution
X, _, y, _ = train_test_split(X, y, train_size=0.1, random_state=42, stratify=y)

In [43]:
# Identify numerical and categorical values

categorical_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

feature_names = numerical_cols + categorical_cols

In [44]:
# Impute the null categorical values
X[categorical_cols] = X[categorical_cols].fillna("NULL")

In [45]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)

## Training Base Model

In [46]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [47]:
encoded_train = preprocessor.fit_transform(X_train)
encoded_test = preprocessor.transform(X_test)

In [48]:
model = RandomForestClassifier(random_state=42)
model.fit(encoded_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [49]:
print("Base model accuracy: ", model.score(encoded_test, y_test))

Base model accuracy:  0.8259979529170931


## Step 3: Run Generalization

First, we train a regular minimizer with `guaranteed_k_anonymity=1` (default behaviour).

In [50]:
train_predicted_y = model.predict(encoded_train)

In [ ]:
default_minimizer = GeneralizeToRepresentative(
    model,
    categorical_features=categorical_cols,
    encoder=preprocessor,
    target_accuracy=0.78,
)


default_minimizer.fit(
    X_train, train_predicted_y, features_names=X_train.columns.tolist()
)

Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.799104
Improving generalizations
Pruned tree to level: 1, new relative accuracy: 0.796545
Pruned tree to level: 2, new relative accuracy: 0.796545
Pruned tree to level: 3, new relative accuracy: 0.797185
Pruned tree to level: 4, new relative accuracy: 0.797185
Pruned tree to level: 5, new relative accuracy: 0.797825
Pruned tree to level: 6, new relative accuracy: 0.795266
Pruned tree to level: 7, new relative accuracy: 0.789507
Pruned tree to level: 8, new relative accuracy: 0.788868
Pruned tree to level: 9, new relative accuracy: 0.784389


,estimator,<apt.utils.mo...x7f1a5b4e75d0>
,target_accuracy,0.78
,cells,"[{'categories': {'education': ['Bachelors', 'Prof-school', ...], 'marital-status': ['Never-married', 'Divorced', ...], 'native-country': ['United-States', 'South', ...], 'occupation': ['Sales', 'Prof-specialty', ...], ...}, 'hist': array([[0., 1.]]), 'id': 10, 'label': [np.int64(1)], ...}, {'categories': {'education': ['Bachelors', 'Prof-school', ...], 'marital-status': ['Widowed'], 'native-country': ['United-States', 'South', ...], 'occupation': ['Sales', 'Prof-specialty', ...], ...}, 'hist': array([[0., 1.]]), 'id': 12, 'label': [np.int64(1)], ...}, ...]"
,categorical_features,"['workclass', 'education', ...]"
,encoder,ColumnTransfo...e-country'])])
,features_to_minimize,"['age', 'workclass', ...]"
,feature_slices,None
,train_only_features_to_minimize,True
,is_regression,False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'


Now we train another minimizer, but this time we force the leaves to be more generalistic.
In theory, this should the accuracy faster, but leave broader ranges.

In [ ]:
stronger_minimizer = GeneralizeToRepresentative(
    model,
    categorical_features=categorical_cols,
    encoder=preprocessor,
    target_accuracy=0.78,
    guaranteed_k_anonymity=10,
)

stronger_minimizer.fit(
    X_train, train_predicted_y, features_names=X_train.columns.tolist()
)

Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.816379
Improving generalizations
Pruned tree to level: 1, new relative accuracy: 0.816379
Pruned tree to level: 2, new relative accuracy: 0.820857
Pruned tree to level: 3, new relative accuracy: 0.820857
Pruned tree to level: 4, new relative accuracy: 0.821497
Pruned tree to level: 5, new relative accuracy: 0.821497
Pruned tree to level: 6, new relative accuracy: 0.823417
Pruned tree to level: 7, new relative accuracy: 0.823417
Pruned tree to level: 8, new relative accuracy: 0.823417
Pruned tree to level: 9, new relative accuracy: 0.834293
Pruned tree to level: 10, new relative accuracy: 0.834293
Pruned tree to level: 11, new relative accuracy: 0.838772
Pruned tree to level: 12, new relative accuracy: 0.834293
Pruned tree to level: 13, new relative accuracy: 0.841971
Pruned tree to level: 14, new relative accuracy: 0.841331
Pruned tree to

,estimator,<apt.utils.mo...x7f1a5b57fa90>
,target_accuracy,0.78
,cells,"[{'categories': {'education': ['Some-college', 'Assoc-acdm', ...], 'marital-status': ['Never-married', 'Married-AF-spouse', ...], 'native-country': ['Hungary', 'Nicaragua', ...], 'occupation': ['NULL', 'Prof-specialty', ...], ...}, 'hist': array([[ 5.47...29.52131855]]), 'id': 2, 'label': [np.int64(1)], ...}, {'categories': {'education': ['Bachelors', 'Prof-school', ...], 'marital-status': ['Never-married', 'Divorced', ...], 'native-country': ['United-States', 'South', ...], 'occupation': ['Sales', 'Exec-managerial', ...], ...}, 'hist': array([[1., 0.]]), 'id': 71, 'label': [np.int64(0)], ...}, ...]"
,categorical_features,"['workclass', 'education', ...]"
,encoder,ColumnTransfo...e-country'])])
,features_to_minimize,"['age', 'workclass', ...]"
,feature_slices,None
,train_only_features_to_minimize,True
,is_regression,False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'


## Step 4: Comparing Accuracy & Disclosure Risk

Let's compare accuracy vs disclosure risk of the baseline model, then when using default generalizer and then when using stronger generalizer!

And as you can below, the stronger generalizer actually outperforms the default generalizer by a long shot!
Not only did the enforced broader DT leaves increased the generalized data "performance generalization" on unseed data in terms of accuracy, it also greatly reduced the disclosure risk (i.e. it provided broader ranges for features) (16% for default generalizer vs 0.4% for stronger generalizer)

In [56]:
baseline_acc = model.score(encoded_test, y_test)
baseline_disclosure_risk = calculate_disclosure_risk(X_test)


print("Baseline model test accuracy: ", baseline_acc)
print("Baseline model test disclosure risk: ", baseline_disclosure_risk)

Baseline model test accuracy:  0.8259979529170931
Baseline model test disclosure risk:  1.0


In [57]:
generalized_X_test = default_minimizer.transform(X_test)
encoded_generalized_x_test = preprocessor.transform(generalized_X_test)

generalized_acc = model.score(encoded_generalized_x_test, y_test)
generalized_disclosure_risk = calculate_disclosure_risk(generalized_X_test)

print("Default Generalized model accuracy: ", generalized_acc)
print("Default Generalized model disclosure risk: ", generalized_disclosure_risk)

Default Generalized model accuracy:  0.77482088024565
Default Generalized model disclosure risk:  0.16376663254861823


In [ ]:
generalized_X_test = stronger_minimizer.transform(X_test)
encoded_generalized_x_test = preprocessor.transform(generalized_X_test)

generalized_acc = model.score(encoded_generalized_x_test, y_test)
generalized_disclosure_risk = calculate_disclosure_risk(generalized_X_test)

print("Stronger Generalized model accuracy: ", generalized_acc)
print("Stronger Generalized model disclosure risk: ", generalized_disclosure_risk)

Stronger Generalized model accuracy:  0.8188331627430911
Stronger Generalized model disclosure risk:  0.0040941658137154556


Also, let's directly compare the feature ranges derives by both generalizers.
You can see how the default minimizer produces highly granular ranges, while the stronger minimzer makes due with only 2 range dividers (or not any at all)!

In [60]:
default_minimizer.generalizations["ranges"]

{'age': [np.float64(28.5),
  np.float64(29.5),
  np.float64(30.5),
  np.float64(31.5),
  np.float64(32.5),
  np.float64(35.0),
  np.float64(35.5),
  np.float64(36.0),
  np.float64(37.5),
  np.float64(38.0),
  np.float64(39.5),
  np.float64(40.0),
  np.float64(41.0),
  np.float64(41.5),
  np.float64(42.0),
  np.float64(42.5),
  np.float64(43.5),
  np.float64(44.0),
  np.float64(44.5),
  np.float64(45.5),
  np.float64(46.5),
  np.float64(47.5),
  np.float64(49.5),
  np.float64(50.0),
  np.float64(51.0),
  np.float64(51.5),
  np.float64(55.5),
  np.float64(56.5),
  np.float64(58.5),
  np.float64(59.0),
  np.float64(59.5),
  np.float64(62.0)],
 'fnlwgt': [np.float64(24864.5),
  np.float64(25243.5),
  np.float64(55620.5),
  np.float64(56149.0),
  np.float64(58186.5),
  np.float64(66630.5),
  np.float64(71458.5),
  np.float64(74158.5),
  np.float64(78376.5),
  np.float64(78479.0),
  np.float64(78516.0),
  np.float64(86291.5),
  np.float64(100891.0),
  np.float64(100957.0),
  np.float64(10456

In [61]:
stronger_minimizer.generalizations["ranges"]

{'age': [np.float64(28.5)],
 'fnlwgt': [],
 'education-num': [np.float64(11.5), np.float64(12.5)],
 'capital-gain': [np.float64(5095.5), np.float64(7055.5)],
 'capital-loss': [],
 'hours-per-week': []}